# Day 01 – Introduction, Pre-processing & Visualization

**Plain-English promise:** give the raw count matrix a spa day so every downstream method sees tidy, normalized numbers. You only need to run the cells below in order.


### What happens today?
1. Load the built-in PBMC 3k dataset (small, perfect for demos).
2. Remove noisy cells/genes, normalize, log-transform, and keep highly variable genes.
3. Build PCA + UMAP so we can literally *see* the cleaned data.


In [ ]:
# Install the needed libraries once (delete the # to run)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


### Step 1 – Import helpful packages

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print('⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.')

try:
    import gseapy as gp
except ImportError:
    gp = None
    print('⚠️ Install gseapy (pip install gseapy) to run the GSEA step.')

try:
    import scvelo as scv
except ImportError:
    scv = None
    print('⚠️ Install scvelo (pip install scvelo) to run the RNA velocity step.')

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


### Step 2 – Load a tiny immune-cell dataset so we have something to clean

In [ ]:
adata = sc.datasets.pbmc3k()  # 3,000 PBMCs shipped with Scanpy
adata.var_names_make_unique()  # keep gene names unique for AnnData rules
adata.layers['counts'] = adata.X.copy()  # stash raw counts for later
adata.obs['source_dataset'] = 'pbmc3k'
adata.raw = adata  # keep an untouched copy for plotting marker genes
print(f'Cells: {adata.n_obs:,} | Genes: {adata.n_vars:,}')


### Step 3 – Clean, normalize, reduce dimensions, and visualize

In [ ]:
adata_day1 = adata.copy()
print('Before QC:', adata_day1.shape)

sc.pp.filter_cells(adata_day1, min_genes=200)  # drop weak cells
sc.pp.filter_genes(adata_day1, min_cells=3)    # drop rarely seen genes
adata_day1.var['mt'] = adata_day1.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_day1, qc_vars=['mt'], inplace=True)
adata_day1 = adata_day1[adata_day1.obs['pct_counts_mt'] < 15, :]

sc.pp.normalize_total(adata_day1, target_sum=1e4)
sc.pp.log1p(adata_day1)
sc.pp.highly_variable_genes(adata_day1, n_top_genes=2000, subset=True)
sc.pp.scale(adata_day1, max_value=10)

sc.tl.pca(adata_day1, n_comps=50)
sc.pp.neighbors(adata_day1, n_neighbors=15)
sc.tl.umap(adata_day1)

print('After QC:', adata_day1.shape)
sc.pl.umap(adata_day1, color=['n_genes', 'pct_counts_mt'], frameon=False)


### Step 4 – Save the cleaned data for Day 02 and beyond

In [ ]:
shared_dir = Path('..') / 'shared_data'
shared_dir.mkdir(parents=True, exist_ok=True)
output_path = shared_dir / 'day01_preprocessed.h5ad'
adata_day1.write(output_path)
print(f'Saved file to {output_path}')
